
# Experiment A - Do layer descriptors improve property prediction?
## A paired, repeated grouped-CV feature-ablation study

**Repository:** [`cookms/squarenet_ml`](https://github.com/cookms/squarenet_ml)  
**Question:** after controlling for composition and inexpensive global crystal descriptors, do square-net detector outputs add reproducible predictive signal for materials properties?

This notebook is designed as a reproducible experiment rather than a leaderboard exercise. It compares five pre-registered feature sets on the **same material cohort, the same grouped splits, the same preprocessing, and the same model specification**. The central outputs are uncertainty intervals and paired split-level differences, not a single best score.

| Experiment element | Pre-registered choice |
|---|---|
| Unit of analysis | One material per row |
| Leakage group | Reduced formula by default; configurable |
| Feature comparison | Baseline; baseline + detector label; baseline + geometry; baseline + chemistry; combined |
| Regression targets | Band gap, energy above hull, formation energy per atom when present |
| Classification targets | Metal/nonmetal, near-hull stability, optional synthesized/experimental status |
| Primary models | Ridge regression and class-balanced logistic regression |
| Validation | Repeated stratified grouped K-fold, generated once per target and reused verbatim |
| Uncertainty | 95% bootstrap interval over repeat-level mean scores |
| Paired comparison | Fold-aligned improvement relative to the baseline; positive always means better |

> **Demo-data guardrail:** the repository currently includes a 16-row material table for pipeline demonstration. The notebook can execute on it as a software smoke test, but it will not present those results as scientific evidence. Point `DATA_PATH` to the full material-level feature table for the actual experiment.



## Scientific hypotheses and interpretation rule

The feature sets are **baseline-anchored ablations**: every extension contains the same baseline, while the combined set contains all families.

- **H0:** layer descriptors do not improve out-of-group property prediction beyond composition and global crystal descriptors.
- **H1-geometry:** geometric square-net descriptors add predictive information.
- **H1-chemistry:** layer chemistry and CrystalNN summaries add predictive information.
- **H1-combined:** the union of detector label, layer geometry, and layer chemistry produces the most reliable improvement.

For each target, the notebook reports the fold-level paired improvement and a 95% interval computed from repeat-level averages. A descriptive evidence label is assigned as follows:

- **Consistent improvement:** interval is entirely above zero and at least 60% of paired folds improve.
- **Inconclusive:** interval overlaps zero.
- **Consistent degradation:** interval is entirely below zero.

This is an estimation-first rubric, not a claim of formal statistical significance. Repeated CV folds are correlated, so the notebook avoids naive independent-fold p-values.


In [5]:

from __future__ import annotations

import hashlib
import importlib.metadata as importlib_metadata
import inspect
import json
import math
import os
import re
import sys
import warnings
from collections import defaultdict
from dataclasses import asdict, dataclass
from pathlib import Path
from typing import Any, Iterable, Mapping, Sequence

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from IPython.display import Markdown, display
from scipy.stats import entropy as scipy_entropy
from sklearn import __version__ as sklearn_version
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression, Ridge
from sklearn.metrics import (
    average_precision_score,
    balanced_accuracy_score,
    f1_score,
    mean_absolute_error,
    mean_squared_error,
    r2_score,
    roc_auc_score,
)
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings('ignore', category=FutureWarning)
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 120)
pd.set_option('display.width', 160)


def find_repo_root(start: Path | None = None) -> Path:
    """Find the repository root or honor SQUARENET_ML_ROOT."""
    override = os.environ.get('SQUARENET_ML_ROOT')
    current = Path(override).expanduser() if override else (start or Path.cwd())
    current = current.resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'requirements.txt').exists() and (candidate / 'squarenet').is_dir():
            return candidate
    raise FileNotFoundError(
        'Could not locate the squarenet_ml repository. Run the notebook inside the clone '
        'or set SQUARENET_ML_ROOT to the repository path.'
    )


@dataclass(frozen=True)
class ExperimentConfig:
    random_state: int = 42
    requested_splits: int = 5
    requested_repeats: int = 10
    demo_splits: int = 3
    demo_repeats: int = 3
    bootstrap_draws: int = 4000
    near_hull_threshold_ev_atom: float = 0.05
    metal_gap_threshold_ev: float = 1e-6
    ridge_alpha: float = 10.0
    logistic_c: float = 1.0
    onehot_min_frequency: int = 5
    demo_onehot_min_frequency: int = 2
    min_materials_for_claim: int = 100
    min_groups_for_claim: int = 50
    grouping_mode: str = 'reduced_formula'
    group_column_override: str | None = None
    data_path_override: str | None = None
    run_experiment: bool = True


CONFIG = ExperimentConfig()
REPO_ROOT = Path(r"C:/Users/mscoo/GIT_projects/Squarenet_ML") #find_repo_root()
OUTPUT_DIR = REPO_ROOT / 'outputs' / 'experiment_A'
FIGURE_DIR = OUTPUT_DIR / 'figures'
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    'figure.figsize': (9.2, 5.2),
    'figure.dpi': 120,
    'savefig.dpi': 180,
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.titleweight': 'bold',
    'axes.labelsize': 10,
    'axes.titlesize': 12,
    'font.size': 10,
    'legend.frameon': False,
    'grid.alpha': 0.25,
})

print(f'Repository root: {REPO_ROOT.name}/')
print(f'Experiment outputs: {OUTPUT_DIR.relative_to(REPO_ROOT)}')
print(f'Python: {sys.version.split()[0]} | pandas: {pd.__version__} | scikit-learn: {sklearn_version}')


Repository root: Squarenet_ML/
Experiment outputs: outputs\experiment_A
Python: 3.10.19 | pandas: 2.3.3 | scikit-learn: 1.7.1



## 1. Load the material-level table

The loader searches for an explicit path first, then common experiment output locations, then the repository demo table. Set either:

```python
CONFIG = ExperimentConfig(data_path_override="/path/to/material_features.parquet")
```

or the environment variable:

```bash
export SQUARENET_ML_EXPERIMENT_A_DATA=/path/to/material_features.parquet
```

The expected grain is **one row per material**. Layer candidates should already be summarized to material-level dominant or aggregate descriptors, as in the repository's `materials.csv` / `ml_materials_demo.csv` outputs.


In [7]:

def discover_data_path(repo_root: Path, override: str | None = None) -> Path:
    env_path = os.environ.get('SQUARENET_ML_EXPERIMENT_A_DATA')
    explicit = override or env_path
    if explicit:
        path = Path(explicit).expanduser().resolve()
        if not path.exists():
            raise FileNotFoundError(f'Configured data path does not exist: {path}')
        return path

    candidates = [
        repo_root / 'outputs' / 'experiment_A' / 'material_features.parquet',
        repo_root / 'outputs' / 'experiment_A' / 'material_features.csv',
        repo_root / 'outputs' / 'property_prediction' / 'material_features.parquet',
        repo_root / 'outputs' / 'property_prediction' / 'material_features.csv',
        repo_root / 'outputs' / 'data_collection_demo' / 'ml_materials_demo.parquet',
        repo_root / 'outputs' / 'data_collection_demo' / 'ml_materials_demo.csv',
        repo_root / 'outputs' / 'data_collection_demo' / 'materials_project_demo' / 'materials.parquet',
        repo_root / 'outputs' / 'data_collection_demo' / 'materials_project_demo' / 'materials.csv',
    ]
    for path in candidates:
        if path.exists():
            return path
    searched = '\n'.join(f'  - {p}' for p in candidates)
    raise FileNotFoundError(f'No material-level data table found. Searched:\n{searched}')


def read_table(path: Path) -> pd.DataFrame:
    suffix = path.suffix.lower()
    if suffix in {'.parquet', '.pq'}:
        return pd.read_parquet(path)
    if suffix in {'.csv', '.txt'}:
        return pd.read_csv(path)
    if suffix in {'.json', '.jsonl'}:
        return pd.read_json(path, lines=suffix == '.jsonl')
    raise ValueError(f'Unsupported table format: {path.suffix}')


DATA_PATH = Path(r"C:\Users\mscoo\GIT_projects\Squarenet_ML\data\materials.csv") #discover_data_path(REPO_ROOT, CONFIG.data_path_override)
df_raw = read_table(DATA_PATH)
df_raw.columns = [str(c).strip() for c in df_raw.columns]

ID_COLUMN_CANDIDATES = ['material_id', 'mp_id', 'task_id', 'id']
ID_COLUMN = next((c for c in ID_COLUMN_CANDIDATES if c in df_raw.columns), None)
if ID_COLUMN is None:
    df_raw = df_raw.copy()
    df_raw['material_id'] = [f'material-{i:07d}' for i in range(len(df_raw))]
    ID_COLUMN = 'material_id'

try:
    DATA_PATH_DISPLAY = DATA_PATH.relative_to(REPO_ROOT)
except ValueError:
    DATA_PATH_DISPLAY = DATA_PATH.name
print(f'Data source: {DATA_PATH_DISPLAY}')
print(f'Shape: {df_raw.shape[0]:,} materials x {df_raw.shape[1]:,} raw columns')
display(df_raw.head(5))


Data source: data\materials.csv
Shape: 154,874 materials x 54 raw columns


,material_id,formula,n_layers_total,n_axes,n_species,n_pass,pass_fraction_total,has_any_pass,dominant_has_pass,dominant_axis,dominant_species,dominant_plane_id,dominant_plane_center_frac,dominant_passes2_fail_reasons,dominant_mean_score,dominant_tol_ratio_any,dominant_nn_intra_min,dominant_min_adj_dist_any_atom,dominant_uv_ang_deg_mean,dominant_uv_ang_err_mean,dominant_uv_len_err_mean,dominant_cnn_in_plane_bond_angle_deg_mean,dominant_cnn_out_of_plane_tilt_angle_deg_mean,dominant_coplane_species_counts_json,dominant_coplane_n_species,dominant_coplane_major_species,dominant_coplane_major_fraction,dominant_coplane_other_species_counts_json,dominant_coplane_other_n_species,dominant_coplane_other_fraction,dominant_cnn_out_of_plane_nn_species,dominant_cnn_out_of_plane_nn_dist,dominant_cnn_out_of_plane_bonded_species_counts_json,dominant_cnn_out_of_plane_bonded_major_species,dominant_cnn_out_of_plane_bonded_major_fraction,dominant_cnn_out_of_plane_bonded_n_species,dominant_square_species_oxi_state_mean,dominant_square_species_oxi_state_std,dominant_has_out_of_plane_same_species_bond,dominant_adj_atom_plane_major_species,dominant_adj_atom_plane_major_fraction,dominant_adj_atom_plane_species_counts_json,dominant_adj_plane_plane_major_species,dominant_adj_plane_plane_major_fraction,dominant_adj_plane_plane_species_counts_json,sg_number,sg_symbol,crystal_system,formula_pretty,formation_energy_per_atom,energy_above_hull,is_stable,band_gap,is_metal
0,mp-1103821,Cu5Sn2Te7,52,3,3,0,0.000000,0,0,b,Cu,1,0.265988,primary_pass_failed,1.984846e-01,1.676686,4.289915,2.558568,89.880927,4.727615e-01,8.547092e-03,NaN,35.463778,"{""Cu"": 5}",1,Cu,1.00,{},0,NaN,Te,2.558568,"{""Te"": 20}",Te,1.0,1,1.2,0.4,0,Te,1.0,"{""Te"": 4}",Sn,1.0,"{""Sn"": 2}",5,C2,Monoclinic,Cu5Sn2Te7,-0.183554,0.0,True,0.0000,True
1,mp-571641,PrCdPd,22,3,3,0,0.000000,0,0,c,Pd,1,0.500000,primary_pass_failed,1.054782e-08,2.748946,7.716443,2.807055,60.000000,3.000000e+01,1.151020e-16,120.0,46.506982,"{""Pd"": 1, ""Pr"": 3}",2,Pr,0.75,"{""Pr"": 3}",1,0.75,Cd,2.807055,"{""Cd"": 6}",Cd,1.0,1,NaN,NaN,0,Cd,0.6,"{""Cd"": 3, ""Pd"": 2}",Cd,0.6,"{""Cd"": 3, ""Pd"": 2}",189,P-62m,Hexagonal,PrCdPd,-0.645004,0.0,True,0.0000,True
2,mp-1076936,ThSiSe,22,3,3,2,0.090909,1,1,c,Si,0,0.000000,NaN,1.000000e+00,0.926302,2.830581,3.055787,90.000000,1.421085e-14,7.844489e-17,90.0,49.080789,"{""Si"": 2}",1,Si,1.00,{},0,NaN,Th,3.055787,"{""Th"": 8}",Th,1.0,1,-2.0,0.0,0,Th,1.0,"{""Th"": 1}",Th,1.0,"{""Th"": 1}",139,I4/mmm,Tetragonal,ThSiSe,-1.358740,0.0,True,0.0000,True
3,mp-1189143,NaNdFeWO6,54,3,5,2,0.037037,1,1,c,Na,6,0.500000,NaN,9.449923e-01,1.645364,3.904552,2.373062,91.665037,1.665037e+00,3.412089e-16,NaN,3.009203,"{""Na"": 2}",1,Na,1.00,{},0,NaN,O,2.373062,"{""O"": 8}",O,1.0,1,NaN,NaN,0,O,1.0,"{""O"": 1}",O,1.0,"{""O"": 1}",4,P2_1,Monoclinic,NaNdFeWO6,-2.548066,0.0,True,2.4146,False
4,mp-1916,YbSb,12,3,2,0,0.000000,0,0,a,Sb,0,0.000000,nn_intra_min_out_of_bounds|coplane_mixed_species,1.000000e+00,1.414214,4.314886,3.051085,90.000000,0.000000e+00,0.000000e+00,90.0,89.999997,"{""Sb"": 2, ""Yb"": 2}",2,Yb,0.50,"{""Yb"": 2}",1,0.50,Yb,3.051085,"{""Yb"": 4}",Yb,1.0,1,-3.0,0.0,0,Yb,0.5,"{""Sb"": 2, ""Yb"": 2}",Yb,0.5,"{""Sb"": 2, ""Yb"": 2}",225,Fm-3m,Cubic,YbSb,-1.271972,0.0,True,0.0000,True



## 2. Canonical targets and provenance

Target construction is explicit and auditable. Direct fields are preferred. Proxy labels are only derived when a direct label is absent:

- **Metal/nonmetal:** direct `is_metal`-style field; otherwise the repository's `target_is_metal_proxy`; otherwise `band_gap <= threshold`.
- **Near-hull stability:** direct label; otherwise `energy_above_hull <= 0.05 eV/atom`.
- **Synthesized/experimental status:** direct positive label; otherwise the inverse of a `theoretical` flag.

All direct target fields and all fields used to derive a proxy are excluded from every feature set. Formation energy is not fabricated when absent.


In [8]:

def first_present(columns: Iterable[str], candidates: Sequence[str]) -> str | None:
    column_set = set(columns)
    return next((name for name in candidates if name in column_set), None)


def coerce_binary(series: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(series):
        return series.astype('Int64')
    numeric = pd.to_numeric(series, errors='coerce')
    if numeric.notna().any():
        unique = set(numeric.dropna().unique())
        if unique.issubset({0, 1}):
            return numeric.round().astype('Int64')
    mapping = {
        'true': 1, 'yes': 1, 'y': 1, '1': 1, 'metal': 1, 'synthesized': 1,
        'experimental': 1, 'stable': 1,
        'false': 0, 'no': 0, 'n': 0, '0': 0, 'nonmetal': 0, 'non-metal': 0,
        'theoretical': 0, 'unstable': 0,
    }
    return series.astype('string').str.strip().str.lower().map(mapping).astype('Int64')


def prepare_targets(frame: pd.DataFrame, config: ExperimentConfig):
    data = frame.copy()
    rows: list[dict[str, Any]] = []
    excluded_sources: set[str] = set()

    regression_specs = {
        'target__band_gap_ev': (
            'Band gap (eV)',
            ['band_gap', 'band_gap_ev', 'gap', 'e_gap'],
        ),
        'target__energy_above_hull_ev_atom': (
            'Energy above hull (eV/atom)',
            ['energy_above_hull', 'e_above_hull', 'energy_above_hull_ev_atom'],
        ),
        'target__formation_energy_per_atom_ev': (
            'Formation energy per atom (eV/atom)',
            [
                'formation_energy_per_atom', 'formation_energy_per_atom_ev',
                'formation_energy_per_atom_ev_atom', 'formation_energy',
            ],
        ),
    }

    for canonical, (label, aliases) in regression_specs.items():
        source = first_present(data.columns, aliases)
        available = source is not None
        if available:
            data[canonical] = pd.to_numeric(data[source], errors='coerce')
            excluded_sources.add(source)
        else:
            data[canonical] = np.nan
        rows.append({
            'target': canonical,
            'display_name': label,
            'task': 'regression',
            'source': source,
            'provenance': 'direct' if source else 'missing',
            'available': available and data[canonical].notna().any(),
        })

    metal_direct = first_present(data.columns, ['is_metal', 'target_is_metal', 'metal'])
    metal_proxy = first_present(data.columns, ['target_is_metal_proxy', 'is_metal_proxy'])
    band_gap_source = first_present(data.columns, ['band_gap', 'band_gap_ev', 'gap', 'e_gap'])
    if metal_direct:
        data['target__is_metal'] = coerce_binary(data[metal_direct])
        metal_source, metal_provenance = metal_direct, 'direct'
        excluded_sources.add(metal_direct)
    elif metal_proxy:
        data['target__is_metal'] = coerce_binary(data[metal_proxy])
        metal_source, metal_provenance = metal_proxy, 'repository proxy'
        excluded_sources.add(metal_proxy)
    elif band_gap_source:
        gap = pd.to_numeric(data[band_gap_source], errors='coerce')
        data['target__is_metal'] = (gap <= config.metal_gap_threshold_ev).where(gap.notna()).astype('Int64')
        metal_source, metal_provenance = band_gap_source, f'derived: gap <= {config.metal_gap_threshold_ev:g} eV'
        excluded_sources.add(band_gap_source)
    else:
        data['target__is_metal'] = pd.Series(pd.NA, index=data.index, dtype='Int64')
        metal_source, metal_provenance = None, 'missing'
    rows.append({
        'target': 'target__is_metal',
        'display_name': 'Metal (1) vs nonmetal (0)',
        'task': 'classification',
        'source': metal_source,
        'provenance': metal_provenance,
        'available': data['target__is_metal'].notna().any(),
    })

    near_direct = first_present(
        data.columns,
        ['target_near_hull_50meV', 'near_hull_50meV', 'is_near_hull', 'target_near_hull'],
    )
    hull_source = first_present(data.columns, ['energy_above_hull', 'e_above_hull', 'energy_above_hull_ev_atom'])
    if near_direct:
        data['target__near_hull'] = coerce_binary(data[near_direct])
        near_source, near_provenance = near_direct, 'direct/repository proxy'
        excluded_sources.add(near_direct)
    elif hull_source:
        hull = pd.to_numeric(data[hull_source], errors='coerce')
        data['target__near_hull'] = (hull <= config.near_hull_threshold_ev_atom).where(hull.notna()).astype('Int64')
        near_source = hull_source
        near_provenance = f'derived: hull <= {config.near_hull_threshold_ev_atom:g} eV/atom'
        excluded_sources.add(hull_source)
    else:
        data['target__near_hull'] = pd.Series(pd.NA, index=data.index, dtype='Int64')
        near_source, near_provenance = None, 'missing'
    rows.append({
        'target': 'target__near_hull',
        'display_name': f'Near-hull stability (<= {config.near_hull_threshold_ev_atom:.2f} eV/atom)',
        'task': 'classification',
        'source': near_source,
        'provenance': near_provenance,
        'available': data['target__near_hull'].notna().any(),
    })

    synth_direct = first_present(
        data.columns,
        ['is_synthesized', 'synthesized', 'is_experimental', 'experimental_status', 'target_is_synthesized'],
    )
    theoretical_source = first_present(data.columns, ['theoretical', 'is_theoretical'])
    if synth_direct:
        data['target__is_synthesized'] = coerce_binary(data[synth_direct])
        synth_source, synth_provenance = synth_direct, 'direct'
        excluded_sources.add(synth_direct)
    elif theoretical_source:
        theoretical = coerce_binary(data[theoretical_source])
        data['target__is_synthesized'] = (1 - theoretical).astype('Int64')
        synth_source, synth_provenance = theoretical_source, 'derived: not theoretical'
        excluded_sources.add(theoretical_source)
    else:
        data['target__is_synthesized'] = pd.Series(pd.NA, index=data.index, dtype='Int64')
        synth_source, synth_provenance = None, 'missing (optional target)'
    rows.append({
        'target': 'target__is_synthesized',
        'display_name': 'Synthesized/experimental status',
        'task': 'classification',
        'source': synth_source,
        'provenance': synth_provenance,
        'available': data['target__is_synthesized'].notna().any(),
    })

    registry = pd.DataFrame(rows)
    registry['n_non_missing'] = registry['target'].map(lambda c: int(data[c].notna().sum()))
    registry['n_unique'] = registry['target'].map(lambda c: int(data[c].nunique(dropna=True)))
    return data, registry, excluded_sources


df, target_registry, target_source_columns = prepare_targets(df_raw, CONFIG)
display(target_registry)


,target,display_name,task,source,provenance,available,n_non_missing,n_unique
0,target__band_gap_ev,Band gap (eV),regression,band_gap,direct,True,153754,56948
1,target__energy_above_hull_ev_atom,Energy above hull (eV/atom),regression,energy_above_hull,direct,True,153754,121069
2,target__formation_energy_per_atom_ev,Formation energy per atom (eV/atom),regression,formation_energy_per_atom,direct,True,153754,153669
3,target__is_metal,Metal (1) vs nonmetal (0),classification,is_metal,direct,True,153686,2
4,target__near_hull,Near-hull stability (<= 0.05 eV/atom),classification,energy_above_hull,derived: hull <= 0.05 eV/atom,True,153754,2
5,target__is_synthesized,Synthesized/experimental status,classification,None,missing (optional target),False,0,0



## 3. Composition baseline and leakage groups

The baseline receives formula-derived composition statistics plus inexpensive global crystal descriptors. When `pymatgen` is available (it is a repository dependency), the composition features include weighted statistics of atomic number, atomic mass, Pauling electronegativity, periodic-table row/group, and atomic radius. A lightweight atomic-number fallback keeps the notebook executable in minimal environments.

The default CV group is **reduced formula**, which keeps polymorphs of the same composition in one fold. Set `group_column_override` for a precomputed prototype/family group, or change `grouping_mode` to `chemical_system` for a stricter element-system holdout.


In [9]:

ELEMENT_SYMBOLS = [
    'H','He','Li','Be','B','C','N','O','F','Ne','Na','Mg','Al','Si','P','S','Cl','Ar','K','Ca',
    'Sc','Ti','V','Cr','Mn','Fe','Co','Ni','Cu','Zn','Ga','Ge','As','Se','Br','Kr','Rb','Sr','Y',
    'Zr','Nb','Mo','Tc','Ru','Rh','Pd','Ag','Cd','In','Sn','Sb','Te','I','Xe','Cs','Ba','La','Ce',
    'Pr','Nd','Pm','Sm','Eu','Gd','Tb','Dy','Ho','Er','Tm','Yb','Lu','Hf','Ta','W','Re','Os','Ir',
    'Pt','Au','Hg','Tl','Pb','Bi','Po','At','Rn','Fr','Ra','Ac','Th','Pa','U','Np','Pu','Am','Cm',
    'Bk','Cf','Es','Fm','Md','No','Lr','Rf','Db','Sg','Bh','Hs','Mt','Ds','Rg','Cn','Nh','Fl','Mc',
    'Lv','Ts','Og'
]
ATOMIC_NUMBER = {symbol: i + 1 for i, symbol in enumerate(ELEMENT_SYMBOLS)}
METALLOIDS = {'B', 'Si', 'Ge', 'As', 'Sb', 'Te', 'Po'}
HALOGENS = {'F', 'Cl', 'Br', 'I', 'At', 'Ts'}
CHALCOGENS = {'O', 'S', 'Se', 'Te', 'Po', 'Lv'}
LANTHANIDES = set(ELEMENT_SYMBOLS[56:71])
ACTINIDES = set(ELEMENT_SYMBOLS[88:103])
NONMETALS = {'H','He','C','N','O','F','Ne','P','S','Cl','Ar','Se','Br','Kr','I','Xe','Rn','Og'}


def fallback_parse_formula(formula: str) -> dict[str, float]:
    """Parse standard nested-parenthesis formulas; pymatgen remains the preferred backend."""
    tokens = re.findall(r'[A-Z][a-z]?|\(|\)|\d+(?:\.\d+)?', str(formula).replace(' ', ''))
    if not tokens:
        raise ValueError(f'Could not parse formula: {formula!r}')

    def parse_group(position: int = 0):
        counts: defaultdict[str, float] = defaultdict(float)
        i = position
        while i < len(tokens):
            token = tokens[i]
            if token == ')':
                return dict(counts), i + 1
            if token == '(':
                nested, i = parse_group(i + 1)
                multiplier = 1.0
                if i < len(tokens) and re.fullmatch(r'\d+(?:\.\d+)?', tokens[i]):
                    multiplier = float(tokens[i])
                    i += 1
                for element, amount in nested.items():
                    counts[element] += amount * multiplier
                continue
            if token not in ATOMIC_NUMBER:
                raise ValueError(f'Unexpected token {token!r} in {formula!r}')
            element = token
            i += 1
            amount = 1.0
            if i < len(tokens) and re.fullmatch(r'\d+(?:\.\d+)?', tokens[i]):
                amount = float(tokens[i])
                i += 1
            counts[element] += amount
        return dict(counts), i

    parsed, end = parse_group(0)
    if end != len(tokens):
        raise ValueError(f'Unbalanced formula: {formula!r}')
    return parsed


def weighted_stats(values: np.ndarray, fractions: np.ndarray, prefix: str) -> dict[str, float]:
    finite = np.isfinite(values) & np.isfinite(fractions)
    values = values[finite]
    fractions = fractions[finite]
    if len(values) == 0 or fractions.sum() <= 0:
        return {f'{prefix}__{name}': np.nan for name in ['mean','std','min','max','range']}
    fractions = fractions / fractions.sum()
    mean = float(np.dot(values, fractions))
    variance = float(np.dot((values - mean) ** 2, fractions))
    return {
        f'{prefix}__mean': mean,
        f'{prefix}__std': math.sqrt(max(variance, 0.0)),
        f'{prefix}__min': float(values.min()),
        f'{prefix}__max': float(values.max()),
        f'{prefix}__range': float(values.max() - values.min()),
    }


try:
    from pymatgen.core import Composition, Element
    COMPOSITION_BACKEND = 'pymatgen'
except Exception:
    Composition = None
    Element = None
    COMPOSITION_BACKEND = 'fallback atomic-number parser'


def composition_record(formula: Any) -> dict[str, Any]:
    if pd.isna(formula):
        return {}
    formula = str(formula)
    if Composition is not None:
        composition = Composition(formula)
        amounts = {el.symbol: float(amount) for el, amount in composition.items()}
        reduced_formula = composition.reduced_formula
        chemical_system = '-'.join(sorted(amounts))
    else:
        amounts = fallback_parse_formula(formula)
        reduced_formula = formula.replace(' ', '')
        chemical_system = '-'.join(sorted(amounts))

    elements = list(amounts)
    raw_amounts = np.array([amounts[e] for e in elements], dtype=float)
    total = float(raw_amounts.sum())
    fractions = raw_amounts / total

    record: dict[str, Any] = {
        'reduced_formula_derived': reduced_formula,
        'chemical_system_derived': chemical_system,
        'comp__n_elements': float(len(elements)),
        'comp__total_atoms_formula': total,
        'comp__stoich_entropy': float(scipy_entropy(fractions)) if len(fractions) else np.nan,
        'comp__max_element_fraction': float(fractions.max()),
        'comp__min_element_fraction': float(fractions.min()),
        'comp__fraction_l2': float(np.sqrt(np.sum(fractions ** 2))),
        'comp__fraction_metal': float(sum(f for e, f in zip(elements, fractions) if e not in NONMETALS and e not in METALLOIDS)),
        'comp__fraction_metalloid': float(sum(f for e, f in zip(elements, fractions) if e in METALLOIDS)),
        'comp__fraction_halogen': float(sum(f for e, f in zip(elements, fractions) if e in HALOGENS)),
        'comp__fraction_chalcogen': float(sum(f for e, f in zip(elements, fractions) if e in CHALCOGENS)),
        'comp__fraction_lanthanide': float(sum(f for e, f in zip(elements, fractions) if e in LANTHANIDES)),
        'comp__fraction_actinide': float(sum(f for e, f in zip(elements, fractions) if e in ACTINIDES)),
    }

    z_values = np.array([ATOMIC_NUMBER.get(e, np.nan) for e in elements], dtype=float)
    record.update(weighted_stats(z_values, fractions, 'comp__atomic_number'))

    if Element is not None:
        property_getters = {
            'atomic_mass': lambda el: float(el.atomic_mass),
            'electronegativity': lambda el: float(el.X) if el.X is not None else np.nan,
            'row': lambda el: float(el.row) if el.row is not None else np.nan,
            'group': lambda el: float(el.group) if el.group is not None else np.nan,
            'atomic_radius': lambda el: float(el.atomic_radius_calculated or el.atomic_radius)
                if (el.atomic_radius_calculated or el.atomic_radius) is not None else np.nan,
        }
        pymatgen_elements = [Element(e) for e in elements]
        for property_name, getter in property_getters.items():
            values = np.array([getter(el) for el in pymatgen_elements], dtype=float)
            record.update(weighted_stats(values, fractions, f'comp__{property_name}'))
    return record


FORMULA_COLUMN = first_present(
    df.columns,
    ['formula', 'pretty_formula', 'reduced_formula', 'composition_reduced', 'formula_pretty'],
)
if FORMULA_COLUMN is None:
    raise KeyError('A formula column is required for the composition baseline and leakage grouping.')

composition_features = pd.DataFrame(
    [composition_record(value) for value in df[FORMULA_COLUMN]],
    index=df.index,
)
df = pd.concat([df, composition_features], axis=1)

if CONFIG.group_column_override:
    if CONFIG.group_column_override not in df.columns:
        raise KeyError(f'Configured group column is missing: {CONFIG.group_column_override}')
    GROUP_COLUMN = CONFIG.group_column_override
elif CONFIG.grouping_mode == 'chemical_system':
    GROUP_COLUMN = 'chemical_system_derived'
else:
    explicit_group = first_present(df.columns, ['group_id', 'composition_group', 'prototype_group'])
    GROUP_COLUMN = explicit_group or 'reduced_formula_derived'

df['cv_group'] = df[GROUP_COLUMN].astype('string').fillna(df[FORMULA_COLUMN].astype('string'))

print(f'Composition backend: {COMPOSITION_BACKEND}')
print(f'Formula column: {FORMULA_COLUMN}')
print(f'CV group source: {GROUP_COLUMN}')
print(f'Unique groups: {df.cv_group.nunique():,} | largest group: {df.cv_group.value_counts().max():,} rows')


C:\Users\mscoo\AppData\Local\Temp\ipykernel_11676\333311676.py:131: UserWarning: No data available for atomic_radius_calculated for Th
  if (el.atomic_radius_calculated or el.atomic_radius) is not None else np.nan,
C:\Users\mscoo\AppData\Local\Temp\ipykernel_11676\333311676.py:130: UserWarning: No data available for atomic_radius_calculated for Th
  'atomic_radius': lambda el: float(el.atomic_radius_calculated or el.atomic_radius)
C:\Users\mscoo\AppData\Local\Temp\ipykernel_11676\333311676.py:131: UserWarning: No data available for atomic_radius_calculated for U
  if (el.atomic_radius_calculated or el.atomic_radius) is not None else np.nan,
C:\Users\mscoo\AppData\Local\Temp\ipykernel_11676\333311676.py:130: UserWarning: No data available for atomic_radius_calculated for U
  'atomic_radius': lambda el: float(el.atomic_radius_calculated or el.atomic_radius)
C:\Users\mscoo\AppData\Local\Temp\ipykernel_11676\333311676.py:131: UserWarning: No data available for atomic_radius_calculated for 

Composition backend: pymatgen
Formula column: formula
CV group source: reduced_formula_derived
Unique groups: 104,839 | largest group: 729 rows



## 4. Compact chemistry summaries from JSON count fields

Raw species-count JSON strings are not modeled directly because one-hot encoding each full dictionary would create brittle, high-cardinality identifiers. Instead, each available count field is converted to stable numeric summaries: total count, species richness, Shannon entropy, and concentration. Existing major-species and major-fraction columns remain available to the chemistry feature family.


In [ ]:

def parse_count_mapping(value: Any) -> dict[str, float]:
    if value is None or (isinstance(value, float) and np.isnan(value)):
        return {}
    if isinstance(value, Mapping):
        mapping = value
    else:
        text = str(value).strip()
        if not text or text.lower() in {'nan', 'none', '<na>'}:
            return {}
        try:
            mapping = json.loads(text)
        except Exception:
            return {}
    cleaned = {}
    for key, amount in mapping.items():
        number = pd.to_numeric(pd.Series([amount]), errors='coerce').iloc[0]
        if pd.notna(number) and float(number) >= 0:
            cleaned[str(key)] = float(number)
    return cleaned


def count_mapping_summary(value: Any, prefix: str) -> dict[str, float]:
    mapping = parse_count_mapping(value)
    if not mapping:
        return {
            f'{prefix}__total': 0.0,
            f'{prefix}__richness': 0.0,
            f'{prefix}__entropy': 0.0,
            f'{prefix}__concentration': 0.0,
            f'{prefix}__max_fraction': np.nan,
        }
    counts = np.array(list(mapping.values()), dtype=float)
    total = float(counts.sum())
    fractions = counts / total if total > 0 else np.zeros_like(counts)
    return {
        f'{prefix}__total': total,
        f'{prefix}__richness': float(len(mapping)),
        f'{prefix}__entropy': float(scipy_entropy(fractions)) if total > 0 else 0.0,
        f'{prefix}__concentration': float(np.sum(fractions ** 2)) if total > 0 else 0.0,
        f'{prefix}__max_fraction': float(fractions.max()) if total > 0 else np.nan,
    }


json_count_columns = [c for c in df.columns if str(c).lower().endswith('counts_json')]
for column in json_count_columns:
    prefix = 'chemjson__' + re.sub(r'_counts_json$', '', column)
    expanded = pd.DataFrame(
        [count_mapping_summary(value, prefix) for value in df[column]],
        index=df.index,
    )
    df = pd.concat([df, expanded], axis=1)

print(f'Expanded {len(json_count_columns)} JSON count fields into compact numeric summaries.')



## 5. Cohort and target audit

The scientific unit is a material, while the statistical unit for leakage control is the configured group. The audit below distinguishes row count from group count, surfaces duplicate IDs, reports target availability, and flags whether the current table is large enough for substantive interpretation.


In [ ]:

def target_audit_table(data: pd.DataFrame, registry: pd.DataFrame) -> pd.DataFrame:
    audited = registry.copy()
    audited['n_non_missing'] = audited['target'].map(lambda c: int(data[c].notna().sum()))
    audited['n_groups'] = audited['target'].map(
        lambda c: int(data.loc[data[c].notna(), 'cv_group'].nunique())
    )
    audited['positive_rate'] = audited.apply(
        lambda row: float(pd.to_numeric(data[row['target']], errors='coerce').mean())
        if row['task'] == 'classification' and row['n_non_missing'] else np.nan,
        axis=1,
    )
    audited['status'] = np.where(audited['available'], 'available', 'skipped')
    return audited


target_registry = target_audit_table(df, target_registry)
duplicate_ids = int(df[ID_COLUMN].duplicated().sum())
missing_formula = int(df[FORMULA_COLUMN].isna().sum())

cohort_card = pd.DataFrame({
    'measure': [
        'materials', 'raw + engineered columns', 'unique CV groups', 'duplicate material IDs',
        'missing formulas', 'materials with detector pass',
    ],
    'value': [
        len(df), len(df.columns), df.cv_group.nunique(), duplicate_ids, missing_formula,
        int(pd.to_numeric(df.get('has_any_pass'), errors='coerce').fillna(0).sum()) if 'has_any_pass' in df else np.nan,
    ],
})

display(cohort_card)
display(target_registry[['display_name','task','source','provenance','n_non_missing','n_groups','positive_rate','status']])

IS_DEMO = (
    len(df) < CONFIG.min_materials_for_claim
    or df.cv_group.nunique() < CONFIG.min_groups_for_claim
)
if IS_DEMO:
    display(Markdown(
        f"> **Smoke-test mode:** only **{len(df):,} materials / {df.cv_group.nunique():,} groups** are available. "
        "The notebook will reduce CV repeats for runtime and will label all predictive comparisons as non-inferential."
    ))
else:
    display(Markdown(
        f"> **Analysis cohort:** **{len(df):,} materials / {df.cv_group.nunique():,} groups**. "
        "The full repeated grouped-CV configuration will be used."
    ))



# Exploratory data analysis

The EDA focuses on questions that affect the validity of Experiment A:

1. Are the requested targets present, sufficiently variable, and reasonably distributed?
2. Are detector-positive and detector-negative materials both represented across the target range?
3. Which feature families are complete enough to compare without target-specific row dropping?
4. Are geometry descriptors strongly redundant, suggesting that regularization is important?


In [ ]:

def save_figure(fig: plt.Figure, name: str) -> None:
    fig.tight_layout()
    fig.savefig(FIGURE_DIR / name, bbox_inches='tight')


available_regression = target_registry.query("available and task == 'regression'")
available_classification = target_registry.query("available and task == 'classification'")

if len(available_regression):
    fig, axes = plt.subplots(1, len(available_regression), figsize=(5.2 * len(available_regression), 4.1), squeeze=False)
    for ax, (_, spec) in zip(axes.ravel(), available_regression.iterrows()):
        values = pd.to_numeric(df[spec.target], errors='coerce').dropna()
        ax.hist(values, bins=min(30, max(6, int(np.sqrt(len(values))))), edgecolor='white', alpha=0.85)
        ax.axvline(values.median(), linestyle='--', linewidth=1.5, label=f'median = {values.median():.3g}')
        ax.set_title(spec.display_name)
        ax.set_xlabel('Target value')
        ax.set_ylabel('Materials')
        ax.legend()
    fig.suptitle('Regression target distributions', y=1.03, fontsize=14, fontweight='bold')
    save_figure(fig, '01_regression_target_distributions.png')
    plt.show()

if len(available_classification):
    fig, axes = plt.subplots(1, len(available_classification), figsize=(4.8 * len(available_classification), 4.1), squeeze=False)
    for ax, (_, spec) in zip(axes.ravel(), available_classification.iterrows()):
        values = pd.to_numeric(df[spec.target], errors='coerce').dropna().astype(int)
        counts = values.value_counts().reindex([0, 1], fill_value=0)
        ax.bar(['0', '1'], counts.values, alpha=0.85)
        for i, value in enumerate(counts.values):
            ax.text(i, value, f'{value:,}', ha='center', va='bottom')
        ax.set_title(spec.display_name)
        ax.set_xlabel('Class')
        ax.set_ylabel('Materials')
    fig.suptitle('Classification target balance', y=1.03, fontsize=14, fontweight='bold')
    save_figure(fig, '02_classification_target_balance.png')
    plt.show()


In [ ]:

if 'has_any_pass' in df.columns:
    pass_flag = pd.to_numeric(df['has_any_pass'], errors='coerce').fillna(0).astype(int)
    summaries = []
    for _, spec in target_registry.query('available').iterrows():
        temp = pd.DataFrame({'pass': pass_flag, 'target': pd.to_numeric(df[spec.target], errors='coerce')}).dropna()
        grouped = temp.groupby('pass')['target'].agg(['count','mean','median','std']).reset_index()
        grouped.insert(0, 'target_name', spec.display_name)
        summaries.append(grouped)
    detector_target_summary = pd.concat(summaries, ignore_index=True) if summaries else pd.DataFrame()
    display(detector_target_summary)

    if len(available_regression):
        fig, axes = plt.subplots(1, len(available_regression), figsize=(5.2 * len(available_regression), 4.2), squeeze=False)
        for ax, (_, spec) in zip(axes.ravel(), available_regression.iterrows()):
            plot_data = pd.DataFrame({
                'has_any_pass': pass_flag,
                'target': pd.to_numeric(df[spec.target], errors='coerce'),
            }).dropna()
            groups = [plot_data.loc[plot_data.has_any_pass == value, 'target'] for value in [0, 1]]
            ax.boxplot(groups, tick_labels=['No pass', 'Any pass'], showfliers=True)
            ax.set_title(spec.display_name)
            ax.set_ylabel('Target value')
        fig.suptitle('Targets by detector pass label', y=1.03, fontsize=14, fontweight='bold')
        save_figure(fig, '03_targets_by_detector_pass.png')
        plt.show()
else:
    display(Markdown('> `has_any_pass` is absent, so detector-label EDA and that feature extension will be limited.'))



## 6. Define feature families

Feature assignment is explicit and pattern-audited. The notebook excludes identifiers, formulas, CV groups, raw JSON dictionaries, detector failure-reason text, canonical targets, direct target fields, and fields used to derive proxy targets.

- **Baseline:** composition descriptors plus space group, crystal system, lattice metrics, density/volume/site-count fields when available.
- **Detector label:** `has_any_pass`, dominant pass state, dominant axis, and dominant species.
- **Layer geometry:** square scores, tolerance ratios, in-plane distances, angle/length errors, and layer/adjacent-plane separations.
- **Layer chemistry:** co-plane and adjacent-plane summaries, oxidation states, CrystalNN bonding/species/angle summaries, and compact JSON-count diversity summaries.


In [ ]:

def ordered_unique(values: Iterable[str]) -> list[str]:
    seen = set()
    result = []
    for value in values:
        if value not in seen:
            seen.add(value)
            result.append(value)
    return result


def columns_matching(columns: Sequence[str], patterns: Sequence[str]) -> list[str]:
    return [c for c in columns if any(re.search(pattern, c, flags=re.IGNORECASE) for pattern in patterns)]


canonical_target_columns = set(target_registry['target'])
all_target_like_columns = {
    c for c in df.columns
    if c in canonical_target_columns
    or c in target_source_columns
    or c.lower().startswith('target_')
    or c.lower().startswith('target__')
}
identifier_columns = {
    ID_COLUMN, FORMULA_COLUMN, 'cv_group', GROUP_COLUMN,
    'reduced_formula_derived', 'chemical_system_derived',
}
raw_unstable_columns = {
    c for c in df.columns
    if c.lower().endswith('counts_json')
    or 'fail_reason' in c.lower()
    or c.lower().endswith('_json')
}
EXCLUDED_COLUMNS = all_target_like_columns | identifier_columns | raw_unstable_columns

composition_columns = [c for c in df.columns if c.startswith('comp__')]

global_exact = [
    'sg_number', 'spacegroup_number', 'space_group_number',
    'sg_symbol', 'spacegroup_symbol', 'space_group_symbol',
    'crystal_system', 'bravais_lattice', 'lattice_type',
    'density', 'density_atomic', 'volume', 'volume_per_atom',
    'nsites', 'n_sites', 'num_sites', 'n_atoms',
    'a', 'b', 'c', 'alpha', 'beta', 'gamma',
    'lattice_a', 'lattice_b', 'lattice_c',
    'lattice_alpha', 'lattice_beta', 'lattice_gamma',
]
global_patterns = [
    r'^(cell|lattice)_(a|b|c|alpha|beta|gamma)(_|$)',
    r'^(volume|density|packing_fraction|atomic_density)(_|$)',
]
global_columns = ordered_unique(
    [c for c in global_exact if c in df.columns]
    + columns_matching(list(df.columns), global_patterns)
)

detector_exact = ['has_any_pass', 'dominant_has_pass', 'dominant_axis', 'dominant_species']
detector_columns = [c for c in detector_exact if c in df.columns]

geometry_patterns = [
    r'(^|_)mean_score($|_)', r'(^|_)score_(mean|min|max|std)($|_)', r'(^|_)tol_ratio',
    r'nn_intra', r'in[_-]?plane.*dist', r'uv_ang', r'uv_len', r'ang_err', r'angle_err',
    r'len_err', r'length_err', r'min_adj_dist', r'layer_sep', r'plane_sep', r'interlayer',
]
geometry_columns = columns_matching(list(df.columns), geometry_patterns)
geometry_columns = [
    c for c in geometry_columns
    if not any(token in c.lower() for token in ['cnn_', 'crystalnn', 'bond_angle', 'tilt_angle'])
]

chemistry_patterns = [
    r'coplane', r'adj_atom_plane', r'adj_plane_plane', r'oxid', r'oxi_', r'cnn_', r'crystalnn',
    r'bonded', r'bond_angle', r'tilt_angle', r'out_of_plane_same_species', r'^chemjson__',
]
chemistry_columns = columns_matching(list(df.columns), chemistry_patterns)

families = {
    'composition': ordered_unique([c for c in composition_columns if c not in EXCLUDED_COLUMNS]),
    'global crystal': ordered_unique([c for c in global_columns if c not in EXCLUDED_COLUMNS]),
    'detector label': ordered_unique([c for c in detector_columns if c not in EXCLUDED_COLUMNS]),
    'layer geometry': ordered_unique([c for c in geometry_columns if c not in EXCLUDED_COLUMNS]),
    'layer chemistry': ordered_unique([c for c in chemistry_columns if c not in EXCLUDED_COLUMNS]),
}

baseline_columns = ordered_unique(families['composition'] + families['global crystal'])
FEATURE_SETS = {
    'Baseline': baseline_columns,
    'Baseline + detector label': ordered_unique(baseline_columns + families['detector label']),
    'Baseline + layer geometry': ordered_unique(baseline_columns + families['layer geometry']),
    'Baseline + layer chemistry': ordered_unique(baseline_columns + families['layer chemistry']),
    'Combined': ordered_unique(
        baseline_columns + families['detector label'] + families['layer geometry'] + families['layer chemistry']
    ),
}
FEATURE_SET_ORDER = list(FEATURE_SETS)

if not FEATURE_SETS['Baseline']:
    raise RuntimeError('The baseline feature set is empty. Check formula and global descriptor columns.')

family_rows = []
for family_name, columns in families.items():
    numeric = [c for c in columns if pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c])]
    categorical = [c for c in columns if c not in numeric]
    family_rows.append({
        'family': family_name,
        'n_columns': len(columns),
        'n_numeric': len(numeric),
        'n_categorical': len(categorical),
        'median_missing_fraction': float(df[columns].isna().mean().median()) if columns else np.nan,
        'example_columns': ', '.join(columns[:7]),
    })
family_manifest = pd.DataFrame(family_rows)

set_manifest = pd.DataFrame([
    {
        'feature_set': name,
        'n_raw_columns': len(columns),
        'n_numeric': sum(pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c]) for c in columns),
        'n_categorical': sum(not (pd.api.types.is_numeric_dtype(df[c]) or pd.api.types.is_bool_dtype(df[c])) for c in columns),
    }
    for name, columns in FEATURE_SETS.items()
])

display(family_manifest)
display(set_manifest)


In [ ]:

# Missingness by feature family
plot_family = family_manifest.dropna(subset=['median_missing_fraction'])
if len(plot_family):
    fig, ax = plt.subplots(figsize=(8.4, 4.5))
    ax.barh(plot_family['family'], plot_family['median_missing_fraction'])
    ax.set_xlim(0, 1)
    ax.set_xlabel('Median fraction missing across family columns')
    ax.set_title('Feature-family completeness')
    save_figure(fig, '04_feature_family_missingness.png')
    plt.show()

# Correlation structure among the most complete geometry features
geometry_numeric = [
    c for c in families['layer geometry']
    if pd.api.types.is_numeric_dtype(df[c]) and df[c].notna().sum() >= max(5, int(0.25 * len(df)))
]
if len(geometry_numeric) >= 2:
    ranked = sorted(geometry_numeric, key=lambda c: (df[c].isna().mean(), -df[c].nunique(dropna=True)))[:16]
    corr = df[ranked].corr(method='spearman')
    fig, ax = plt.subplots(figsize=(max(7, 0.55 * len(ranked)), max(6, 0.48 * len(ranked))))
    image = ax.imshow(corr, vmin=-1, vmax=1, cmap='coolwarm', aspect='auto')
    ax.set_xticks(range(len(ranked)), labels=ranked, rotation=75, ha='right')
    ax.set_yticks(range(len(ranked)), labels=ranked)
    ax.set_title('Spearman correlation among layer-geometry descriptors')
    fig.colorbar(image, ax=ax, shrink=0.8, label='Spearman rho')
    save_figure(fig, '05_geometry_correlation.png')
    plt.show()
else:
    display(Markdown('> Fewer than two sufficiently complete numeric geometry features are available for a correlation audit.'))



# Experimental protocol

### Fair-comparison controls

- The target-specific cohort is selected **once** using target availability only; rows are not dropped because a feature is missing.
- Grouped splits are generated **once per target** and then reused for every feature set.
- Numeric preprocessing is median imputation plus missing indicators and standardization.
- Categorical preprocessing is most-frequent imputation plus fold-local one-hot encoding with unknown-category handling.
- The model specification and regularization are fixed before seeing feature-set results.
- Every target-like/source column is excluded from all predictors.

### Why a regularized linear probe?

Ridge and logistic regression are deliberately used as controlled information probes. They are fast enough for repeated grouped CV, support identical mixed-type preprocessing, and minimize the risk that hyperparameter search or different nonlinear interactions masquerade as descriptor value. A later robustness notebook can repeat the same frozen splits with a nonlinear estimator.

### Metrics

- Regression: MAE (primary), RMSE, and R-squared.
- Classification: ROC AUC (primary when defined), average precision, balanced accuracy, and F1.
- Paired differences are re-oriented so **positive always means the candidate feature set is better**.


In [ ]:

def regression_strata(y: pd.Series, max_bins: int = 10) -> pd.Series:
    values = pd.to_numeric(y, errors='coerce')
    n = len(values)
    unique = values.nunique(dropna=True)
    bins = min(max_bins, max(2, n // 20), unique)
    if bins < 2:
        return pd.Series(np.zeros(n, dtype=int), index=y.index)
    ranked = values.rank(method='first')
    return pd.qcut(ranked, q=int(bins), labels=False, duplicates='drop').astype(int)


def valid_classification_split(y_train: pd.Series, y_test: pd.Series) -> bool:
    return y_train.nunique() >= 2 and y_test.nunique() >= 2


def generate_target_splits(
    data: pd.DataFrame,
    target: str,
    task: str,
    requested_splits: int,
    repeats: int,
    random_state: int,
) -> tuple[pd.DataFrame, list[dict[str, Any]]]:
    cohort = data.loc[data[target].notna(), [ID_COLUMN, 'cv_group', target]].copy()
    cohort[target] = pd.to_numeric(cohort[target], errors='coerce')
    cohort = cohort.dropna(subset=[target]).reset_index().rename(columns={'index': 'source_index'})
    n_groups = cohort.cv_group.nunique()
    max_splits = min(requested_splits, n_groups)
    if task == 'classification':
        class_counts = cohort[target].astype(int).value_counts()
        max_splits = min(max_splits, int(class_counts.min())) if len(class_counts) >= 2 else 1
    if max_splits < 2:
        raise ValueError(f'{target}: fewer than two viable grouped folds.')

    strata = cohort[target].astype(int) if task == 'classification' else regression_strata(cohort[target])
    chosen_splits = None
    all_splits: list[dict[str, Any]] = []

    for n_splits in range(max_splits, 1, -1):
        candidate: list[dict[str, Any]] = []
        valid = True
        for repeat in range(repeats):
            splitter = StratifiedGroupKFold(
                n_splits=n_splits,
                shuffle=True,
                random_state=random_state + repeat,
            )
            try:
                split_iter = splitter.split(cohort, strata, groups=cohort['cv_group'])
                repeat_splits = list(split_iter)
            except ValueError:
                valid = False
                break
            for fold, (train_pos, test_pos) in enumerate(repeat_splits):
                train_groups = set(cohort.iloc[train_pos]['cv_group'])
                test_groups = set(cohort.iloc[test_pos]['cv_group'])
                if train_groups & test_groups:
                    raise AssertionError('Group leakage detected while constructing CV splits.')
                if task == 'classification' and not valid_classification_split(
                    cohort.iloc[train_pos][target], cohort.iloc[test_pos][target]
                ):
                    valid = False
                    break
                candidate.append({
                    'target': target,
                    'repeat': repeat,
                    'fold': fold,
                    'split_id': f'{target}__r{repeat:02d}_f{fold:02d}',
                    'train_source_index': cohort.iloc[train_pos]['source_index'].to_numpy(),
                    'test_source_index': cohort.iloc[test_pos]['source_index'].to_numpy(),
                    'n_train': len(train_pos),
                    'n_test': len(test_pos),
                    'n_train_groups': len(train_groups),
                    'n_test_groups': len(test_groups),
                })
            if not valid:
                break
        if valid:
            chosen_splits = n_splits
            all_splits = candidate
            break

    if chosen_splits is None:
        raise ValueError(f'{target}: could not construct grouped folds with valid train/test targets.')

    manifest_rows = []
    for split in all_splits:
        test_ids = data.loc[split['test_source_index'], ID_COLUMN]
        test_groups = data.loc[split['test_source_index'], 'cv_group']
        for material_id, group_id in zip(test_ids, test_groups):
            manifest_rows.append({
                'target': target,
                'repeat': split['repeat'],
                'fold': split['fold'],
                'split_id': split['split_id'],
                'material_id': material_id,
                'cv_group': group_id,
                'role': 'test',
            })
    return pd.DataFrame(manifest_rows), all_splits


In [ ]:

def compatible_one_hot(min_frequency: int) -> OneHotEncoder:
    kwargs = {
        'handle_unknown': 'ignore',
        'min_frequency': min_frequency,
    }
    signature = inspect.signature(OneHotEncoder)
    if 'sparse_output' in signature.parameters:
        kwargs['sparse_output'] = True
    else:
        kwargs['sparse'] = True
    return OneHotEncoder(**kwargs)


def make_preprocessor(X: pd.DataFrame, min_frequency: int) -> ColumnTransformer:
    numeric_columns = [
        c for c in X.columns
        if pd.api.types.is_numeric_dtype(X[c]) or pd.api.types.is_bool_dtype(X[c])
    ]
    categorical_columns = [c for c in X.columns if c not in numeric_columns]

    transformers = []
    if numeric_columns:
        numeric_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='median', add_indicator=True, keep_empty_features=True)),
            ('scaler', StandardScaler()),
        ])
        transformers.append(('numeric', numeric_pipeline, numeric_columns))
    if categorical_columns:
        categorical_pipeline = Pipeline([
            ('imputer', SimpleImputer(strategy='most_frequent', keep_empty_features=True)),
            ('onehot', compatible_one_hot(min_frequency=min_frequency)),
        ])
        transformers.append(('categorical', categorical_pipeline, categorical_columns))
    if not transformers:
        raise ValueError('No usable columns were supplied to the preprocessor.')
    return ColumnTransformer(
        transformers=transformers,
        remainder='drop',
        sparse_threshold=1.0,
        verbose_feature_names_out=False,
    )


def make_model(task: str, X: pd.DataFrame, config: ExperimentConfig, min_frequency: int) -> Pipeline:
    preprocess = make_preprocessor(X, min_frequency=min_frequency)
    if task == 'regression':
        estimator = TransformedTargetRegressor(
            regressor=Ridge(alpha=config.ridge_alpha, solver='lsqr'),
            transformer=StandardScaler(),
        )
    else:
        estimator = LogisticRegression(
            C=config.logistic_c,
            class_weight='balanced',
            solver='liblinear',
            max_iter=5000,
            random_state=config.random_state,
        )
    return Pipeline([('preprocess', preprocess), ('model', estimator)])


def safe_regression_metrics(y_true: np.ndarray, y_pred: np.ndarray) -> dict[str, float]:
    metrics = {
        'mae': mean_absolute_error(y_true, y_pred),
        'rmse': math.sqrt(mean_squared_error(y_true, y_pred)),
        'r2': r2_score(y_true, y_pred) if len(y_true) >= 2 else np.nan,
    }
    return {key: float(value) for key, value in metrics.items()}


def safe_classification_metrics(y_true: np.ndarray, y_score: np.ndarray) -> dict[str, float]:
    y_pred = (y_score >= 0.5).astype(int)
    both_classes = np.unique(y_true).size == 2
    return {
        'roc_auc': float(roc_auc_score(y_true, y_score)) if both_classes else np.nan,
        'average_precision': float(average_precision_score(y_true, y_score)) if both_classes else np.nan,
        'balanced_accuracy': float(balanced_accuracy_score(y_true, y_pred)),
        'f1': float(f1_score(y_true, y_pred, zero_division=0)),
    }


def evaluate_feature_sets(
    data: pd.DataFrame,
    target_registry: pd.DataFrame,
    feature_sets: dict[str, list[str]],
    config: ExperimentConfig,
    is_demo: bool,
):
    score_rows: list[dict[str, Any]] = []
    prediction_rows: list[dict[str, Any]] = []
    split_manifests: list[pd.DataFrame] = []
    failure_rows: list[dict[str, Any]] = []

    repeats = config.demo_repeats if is_demo else config.requested_repeats
    requested_splits = config.demo_splits if is_demo else config.requested_splits
    min_frequency = config.demo_onehot_min_frequency if is_demo else config.onehot_min_frequency

    for _, spec in target_registry.query('available').iterrows():
        target = spec['target']
        task = spec['task']
        print(f'\n{spec.display_name}: building shared grouped splits...')
        try:
            split_manifest, splits = generate_target_splits(
                data=data,
                target=target,
                task=task,
                requested_splits=requested_splits,
                repeats=repeats,
                random_state=config.random_state,
            )
        except Exception as exc:
            failure_rows.append({'target': target, 'stage': 'split generation', 'error': repr(exc)})
            print(f'  skipped: {exc}')
            continue
        split_manifests.append(split_manifest)
        print(f'  {len(splits)} paired splits ({repeats} repeats x {len(splits) // repeats} folds)')

        for feature_set_name in FEATURE_SET_ORDER:
            columns = feature_sets[feature_set_name]
            print(f'  fitting {feature_set_name} ({len(columns)} raw columns)')
            for split in splits:
                train_index = split['train_source_index']
                test_index = split['test_source_index']
                X_train = data.loc[train_index, columns].copy()
                X_test = data.loc[test_index, columns].copy()
                y_train = pd.to_numeric(data.loc[train_index, target], errors='coerce')
                y_test = pd.to_numeric(data.loc[test_index, target], errors='coerce')

                try:
                    model = make_model(task, X_train, config, min_frequency=min_frequency)
                    model.fit(X_train, y_train)
                    if task == 'regression':
                        y_pred = np.asarray(model.predict(X_test), dtype=float)
                        metric_values = safe_regression_metrics(y_test.to_numpy(dtype=float), y_pred)
                        y_score = np.full(len(y_pred), np.nan)
                    else:
                        y_score = np.asarray(model.predict_proba(X_test)[:, 1], dtype=float)
                        y_pred = (y_score >= 0.5).astype(int)
                        metric_values = safe_classification_metrics(y_test.to_numpy(dtype=int), y_score)

                    try:
                        transformed_features = len(model.named_steps['preprocess'].get_feature_names_out())
                    except Exception:
                        transformed_features = np.nan

                    for metric, score in metric_values.items():
                        score_rows.append({
                            'target': target,
                            'target_name': spec.display_name,
                            'task': task,
                            'feature_set': feature_set_name,
                            'repeat': split['repeat'],
                            'fold': split['fold'],
                            'split_id': split['split_id'],
                            'metric': metric,
                            'score': score,
                            'n_train': split['n_train'],
                            'n_test': split['n_test'],
                            'n_train_groups': split['n_train_groups'],
                            'n_test_groups': split['n_test_groups'],
                            'n_raw_features': len(columns),
                            'n_transformed_features': transformed_features,
                        })

                    for row_position, source_index in enumerate(test_index):
                        prediction_rows.append({
                            'target': target,
                            'target_name': spec.display_name,
                            'task': task,
                            'feature_set': feature_set_name,
                            'repeat': split['repeat'],
                            'fold': split['fold'],
                            'split_id': split['split_id'],
                            'source_index': int(source_index),
                            'material_id': data.loc[source_index, ID_COLUMN],
                            'cv_group': data.loc[source_index, 'cv_group'],
                            'formula': data.loc[source_index, FORMULA_COLUMN],
                            'y_true': float(y_test.iloc[row_position]),
                            'y_pred': float(y_pred[row_position]),
                            'y_score': float(y_score[row_position]) if np.isfinite(y_score[row_position]) else np.nan,
                        })
                except Exception as exc:
                    failure_rows.append({
                        'target': target,
                        'feature_set': feature_set_name,
                        'split_id': split['split_id'],
                        'stage': 'fit/evaluate',
                        'error': repr(exc),
                    })

    scores = pd.DataFrame(score_rows)
    predictions = pd.DataFrame(prediction_rows)
    split_manifest = pd.concat(split_manifests, ignore_index=True) if split_manifests else pd.DataFrame()
    failures = pd.DataFrame(failure_rows)
    return scores, predictions, split_manifest, failures


In [ ]:

if CONFIG.run_experiment:
    fold_scores, fold_predictions, split_manifest, failures = evaluate_feature_sets(
        data=df,
        target_registry=target_registry,
        feature_sets=FEATURE_SETS,
        config=CONFIG,
        is_demo=IS_DEMO,
    )
else:
    fold_scores = pd.DataFrame()
    fold_predictions = pd.DataFrame()
    split_manifest = pd.DataFrame()
    failures = pd.DataFrame()

print(f'\nCompleted score rows: {len(fold_scores):,}')
print(f'Completed prediction rows: {len(fold_predictions):,}')
if len(failures):
    display(Markdown(f'> **Evaluation warnings:** {len(failures)} fit or split operations failed.'))
    display(failures.head(20))
else:
    print('No split or model failures recorded.')



## 7. Uncertainty and paired fold-level differences

Raw fold scores are retained, but uncertainty is summarized over **repeat-level means**. This blocks the bootstrap at the repeat level and avoids treating all overlapping folds across repeats as independent observations.

For paired comparisons, each candidate score is joined to the baseline score by target, metric, repeat, fold, and split ID. Error metrics are sign-flipped so that positive improvement always has the same interpretation.


In [ ]:

LOWER_IS_BETTER = {'mae', 'rmse', 'log_loss', 'brier'}
PRIMARY_METRIC = {'regression': 'mae', 'classification': 'roc_auc'}


def stable_seed(*parts: Any, base: int = 0) -> int:
    payload = '||'.join(map(str, parts)).encode('utf-8')
    return base + int.from_bytes(hashlib.sha256(payload).digest()[:4], 'big') % 100000


def bootstrap_mean_ci(values: Sequence[float], draws: int, seed: int, confidence: float = 0.95):
    array = np.asarray(values, dtype=float)
    array = array[np.isfinite(array)]
    if len(array) == 0:
        return np.nan, np.nan
    if len(array) == 1:
        return float(array[0]), float(array[0])
    rng = np.random.default_rng(seed)
    sampled = rng.choice(array, size=(draws, len(array)), replace=True).mean(axis=1)
    alpha = (1 - confidence) / 2
    return tuple(np.quantile(sampled, [alpha, 1 - alpha]).astype(float))


def summarize_scores(scores: pd.DataFrame, config: ExperimentConfig) -> pd.DataFrame:
    if scores.empty:
        return pd.DataFrame()
    repeat_scores = (
        scores.groupby(['target','target_name','task','feature_set','metric','repeat'], as_index=False)
        .agg(repeat_mean_score=('score','mean'))
    )
    rows = []
    for key, group in repeat_scores.groupby(['target','target_name','task','feature_set','metric'], sort=False):
        target, target_name, task, feature_set, metric = key
        ci_low, ci_high = bootstrap_mean_ci(
            group.repeat_mean_score,
            draws=config.bootstrap_draws,
            seed=stable_seed('score', target, feature_set, metric, base=config.random_state),
        )
        fold_group = scores[
            (scores.target == target)
            & (scores.feature_set == feature_set)
            & (scores.metric == metric)
        ]
        rows.append({
            'target': target,
            'target_name': target_name,
            'task': task,
            'feature_set': feature_set,
            'metric': metric,
            'mean_score': float(group.repeat_mean_score.mean()),
            'repeat_std': float(group.repeat_mean_score.std(ddof=1)) if len(group) > 1 else np.nan,
            'ci_low': ci_low,
            'ci_high': ci_high,
            'n_repeats': int(group.repeat.nunique()),
            'n_fold_scores': int(fold_group.score.notna().sum()),
        })
    return pd.DataFrame(rows)


def paired_against_baseline(scores: pd.DataFrame, baseline: str = 'Baseline') -> pd.DataFrame:
    if scores.empty:
        return pd.DataFrame()
    keys = ['target','target_name','task','repeat','fold','split_id','metric']
    reference = scores.loc[scores.feature_set == baseline, keys + ['score']].rename(columns={'score':'baseline_score'})
    candidates = scores.loc[scores.feature_set != baseline, keys + ['feature_set','score']].rename(columns={'score':'candidate_score'})
    paired = candidates.merge(reference, on=keys, how='inner', validate='many_to_one')
    direction = np.where(paired.metric.isin(LOWER_IS_BETTER), -1.0, 1.0)
    paired['improvement'] = direction * (paired['candidate_score'] - paired['baseline_score'])
    paired['improved'] = paired['improvement'] > 0
    return paired


def summarize_paired(paired: pd.DataFrame, config: ExperimentConfig) -> pd.DataFrame:
    if paired.empty:
        return pd.DataFrame()
    repeat_delta = (
        paired.groupby(['target','target_name','task','feature_set','metric','repeat'], as_index=False)
        .agg(repeat_mean_improvement=('improvement','mean'))
    )
    rows = []
    for key, group in repeat_delta.groupby(['target','target_name','task','feature_set','metric'], sort=False):
        target, target_name, task, feature_set, metric = key
        ci_low, ci_high = bootstrap_mean_ci(
            group.repeat_mean_improvement,
            draws=config.bootstrap_draws,
            seed=stable_seed('paired', target, feature_set, metric, base=config.random_state),
        )
        fold_group = paired[
            (paired.target == target)
            & (paired.feature_set == feature_set)
            & (paired.metric == metric)
        ]
        mean_improvement = float(group.repeat_mean_improvement.mean())
        win_rate = float(fold_group.improved.mean())
        if ci_low > 0 and win_rate >= 0.60:
            evidence = 'consistent improvement'
        elif ci_high < 0:
            evidence = 'consistent degradation'
        else:
            evidence = 'inconclusive'
        rows.append({
            'target': target,
            'target_name': target_name,
            'task': task,
            'feature_set': feature_set,
            'metric': metric,
            'mean_improvement': mean_improvement,
            'median_fold_improvement': float(fold_group.improvement.median()),
            'ci_low': ci_low,
            'ci_high': ci_high,
            'fold_win_rate': win_rate,
            'n_paired_folds': int(fold_group.improvement.notna().sum()),
            'n_repeats': int(group.repeat.nunique()),
            'evidence': evidence,
        })
    return pd.DataFrame(rows)


score_summary = summarize_scores(fold_scores, CONFIG)
paired_fold_differences = paired_against_baseline(fold_scores)
paired_summary = summarize_paired(paired_fold_differences, CONFIG)

primary_rows = []
for _, spec in target_registry.query('available').iterrows():
    metric = PRIMARY_METRIC[spec.task]
    subset = score_summary[(score_summary.target == spec.target) & (score_summary.metric == metric)]
    if subset.empty and spec.task == 'classification':
        metric = 'balanced_accuracy'
        subset = score_summary[(score_summary.target == spec.target) & (score_summary.metric == metric)]
    primary_rows.append(subset)
primary_score_summary = pd.concat(primary_rows, ignore_index=True) if primary_rows else pd.DataFrame()

paired_primary_rows = []
for _, spec in target_registry.query('available').iterrows():
    metric = PRIMARY_METRIC[spec.task]
    subset = paired_summary[(paired_summary.target == spec.target) & (paired_summary.metric == metric)]
    if subset.empty and spec.task == 'classification':
        metric = 'balanced_accuracy'
        subset = paired_summary[(paired_summary.target == spec.target) & (paired_summary.metric == metric)]
    paired_primary_rows.append(subset)
paired_primary = pd.concat(paired_primary_rows, ignore_index=True) if paired_primary_rows else pd.DataFrame()

primary_score_display = primary_score_summary.copy()
primary_score_display['display_rank'] = np.where(
    primary_score_display.metric.isin(LOWER_IS_BETTER),
    primary_score_display.mean_score,
    -primary_score_display.mean_score,
)
display(primary_score_display.sort_values(['target_name','display_rank']).drop(columns='display_rank'))
display(paired_primary.sort_values(['target_name','mean_improvement'], ascending=[True, False]))


In [ ]:

# Primary metric score plots
for target, group in primary_score_summary.groupby('target', sort=False):
    group = group.set_index('feature_set').reindex(FEATURE_SET_ORDER).dropna(subset=['mean_score']).reset_index()
    if group.empty:
        continue
    task = group.task.iloc[0]
    metric = group.metric.iloc[0]
    fig, ax = plt.subplots(figsize=(8.8, 4.8))
    y = np.arange(len(group))
    lower = group.mean_score - group.ci_low
    upper = group.ci_high - group.mean_score
    ax.errorbar(group.mean_score, y, xerr=[lower, upper], fmt='o', capsize=4)
    ax.set_yticks(y, labels=group.feature_set)
    ax.invert_yaxis()
    ax.set_xlabel(f'{metric} (95% repeat-block bootstrap interval)')
    direction = 'lower is better' if metric in LOWER_IS_BETTER else 'higher is better'
    ax.set_title(f'{group.target_name.iloc[0]}: {metric} ({direction})')
    ax.grid(axis='x')
    save_figure(fig, f'06_score_{target}.png')
    plt.show()

# Paired fold improvement plots
for target, group in paired_fold_differences.groupby('target', sort=False):
    task = group.task.iloc[0]
    metric = PRIMARY_METRIC[task]
    plot_data = group[group.metric == metric].copy()
    if plot_data.empty and task == 'classification':
        metric = 'balanced_accuracy'
        plot_data = group[group.metric == metric].copy()
    if plot_data.empty:
        continue
    candidates = [name for name in FEATURE_SET_ORDER if name != 'Baseline' and name in set(plot_data.feature_set)]
    fig, ax = plt.subplots(figsize=(9.0, 5.2))
    rng = np.random.default_rng(CONFIG.random_state)
    for position, name in enumerate(candidates):
        values = plot_data.loc[plot_data.feature_set == name, 'improvement'].dropna().to_numpy()
        jitter = rng.normal(0, 0.045, size=len(values))
        ax.scatter(values, np.full(len(values), position) + jitter, s=28, alpha=0.65)
        if len(values):
            ax.plot([np.median(values)], [position], marker='D', markersize=7)
    ax.axvline(0, color='black', linewidth=1, linestyle='--')
    ax.set_yticks(range(len(candidates)), labels=candidates)
    ax.invert_yaxis()
    ax.set_xlabel(f'Paired improvement in {metric} vs Baseline (positive = better)')
    ax.set_title(f'{plot_data.target_name.iloc[0]}: fold-aligned paired differences')
    ax.grid(axis='x')
    save_figure(fig, f'07_paired_delta_{target}.png')
    plt.show()


In [ ]:

# Cross-target improvement matrix for the primary metric
if not paired_primary.empty:
    matrix = paired_primary.pivot(index='target_name', columns='feature_set', values='mean_improvement')
    matrix = matrix.reindex(columns=[name for name in FEATURE_SET_ORDER if name != 'Baseline'])
    fig, ax = plt.subplots(figsize=(10.2, max(3.8, 0.7 * len(matrix))))
    finite = matrix.to_numpy(dtype=float)
    limit = np.nanmax(np.abs(finite)) if np.isfinite(finite).any() else 1.0
    limit = max(limit, 1e-12)
    image = ax.imshow(matrix, cmap='coolwarm', vmin=-limit, vmax=limit, aspect='auto')
    ax.set_xticks(range(len(matrix.columns)), labels=matrix.columns, rotation=35, ha='right')
    ax.set_yticks(range(len(matrix.index)), labels=matrix.index)
    for row in range(matrix.shape[0]):
        for column in range(matrix.shape[1]):
            value = matrix.iloc[row, column]
            if pd.notna(value):
                ax.text(column, row, f'{value:+.3g}', ha='center', va='center', fontsize=9)
    ax.set_title('Mean paired primary-metric improvement vs Baseline')
    fig.colorbar(image, ax=ax, shrink=0.8, label='Positive = better')
    save_figure(fig, '08_primary_metric_improvement_matrix.png')
    plt.show()



## 8. Out-of-fold error analysis

Repeated out-of-fold predictions are averaged per material and feature set. For regression, the material-level improvement is the reduction in absolute error. For classification, it is the reduction in squared probability error. This identifies where combined layer descriptors help or hurt and supports domain review of specific chemistries rather than only aggregate metrics.


In [ ]:

def material_error_analysis(predictions: pd.DataFrame) -> pd.DataFrame:
    if predictions.empty:
        return pd.DataFrame()
    aggregated = (
        predictions.groupby(
            ['target','target_name','task','feature_set','material_id','cv_group','formula'],
            as_index=False,
        )
        .agg(y_true=('y_true','first'), mean_y_pred=('y_pred','mean'), mean_y_score=('y_score','mean'), n_oof=('repeat','nunique'))
    )
    regression = aggregated[aggregated.task == 'regression'].copy()
    regression['error'] = (regression.y_true - regression.mean_y_pred).abs()
    classification = aggregated[aggregated.task == 'classification'].copy()
    classification['error'] = (classification.y_true - classification.mean_y_score) ** 2
    regression['mean_prediction'] = regression['mean_y_pred']
    classification['mean_prediction'] = classification['mean_y_score']
    combined = pd.concat([regression, classification], ignore_index=True)

    baseline = combined[combined.feature_set == 'Baseline'][
        ['target','material_id','error']
    ].rename(columns={'error':'baseline_error'})
    candidates = combined[combined.feature_set != 'Baseline'].copy()
    candidates = candidates.merge(baseline, on=['target','material_id'], how='left', validate='many_to_one')
    candidates['material_error_improvement'] = candidates['baseline_error'] - candidates['error']
    return candidates


material_errors = material_error_analysis(fold_predictions)
combined_material_errors = material_errors[material_errors.feature_set == 'Combined'].copy()

if not combined_material_errors.empty:
    for target, group in combined_material_errors.groupby('target', sort=False):
        display(Markdown(f"### {group.target_name.iloc[0]} - materials most helped and most hurt by Combined"))
        columns = ['material_id','formula','cv_group','y_true','mean_prediction','baseline_error','error','material_error_improvement']
        helped = group.nlargest(min(7, len(group)), 'material_error_improvement')[columns]
        hurt = group.nsmallest(min(7, len(group)), 'material_error_improvement')[columns]
        display(Markdown('**Largest error reductions**'))
        display(helped)
        display(Markdown('**Largest error increases**'))
        display(hurt)



# Automated experiment readout

The readout below is generated only from the paired primary-metric summaries. In smoke-test mode it deliberately uses the phrase **software result**, not scientific conclusion.


In [ ]:

def automated_readout(
    paired_primary: pd.DataFrame,
    registry: pd.DataFrame,
    is_demo: bool,
) -> str:
    if paired_primary.empty:
        return 'No paired primary-metric results were available.'
    lines = []
    for target, group in paired_primary.groupby('target', sort=False):
        target_name = group.target_name.iloc[0]
        combined = group[group.feature_set == 'Combined']
        best = group.sort_values('mean_improvement', ascending=False).iloc[0]
        if combined.empty:
            combined_text = 'Combined was not estimable.'
        else:
            row = combined.iloc[0]
            combined_text = (
                f"Combined vs Baseline: {row.mean_improvement:+.3g} "
                f"(95% interval {row.ci_low:+.3g} to {row.ci_high:+.3g}; "
                f"fold win rate {row.fold_win_rate:.0%}; {row.evidence})."
            )
        lines.append(
            f"- **{target_name}:** {combined_text} "
            + (
                f"Largest mean improvement: **{best.feature_set}** ({best.mean_improvement:+.3g})."
                if best.mean_improvement > 0 else
                f"No extension improved on average; least negative: **{best.feature_set}** ({best.mean_improvement:+.3g})."
            )
        )
    prefix = (
        '**Smoke-test readout - do not interpret scientifically.** The current cohort is below the pre-set size guardrail.\n\n'
        if is_demo else
        '**Experiment A readout.** Results use repeated grouped CV and repeat-blocked uncertainty.\n\n'
    )
    return prefix + '\n'.join(lines)


display(Markdown(automated_readout(paired_primary, target_registry, IS_DEMO)))



## 9. Persist reproducibility artifacts

The notebook writes tidy, versionable outputs rather than relying on rendered cells alone:

- fold-level scores and predictions;
- exact test-fold assignments;
- paired fold differences and repeat-blocked summaries;
- feature-family and feature-set manifests;
- target provenance and experiment configuration;
- a SHA-256 hash of the input table;
- publication-ready figures.


In [ ]:

def sha256_file(path: Path, chunk_size: int = 1024 * 1024) -> str:
    digest = hashlib.sha256()
    with path.open('rb') as handle:
        while chunk := handle.read(chunk_size):
            digest.update(chunk)
    return digest.hexdigest()


def package_version(name: str) -> str | None:
    try:
        return importlib_metadata.version(name)
    except importlib_metadata.PackageNotFoundError:
        return None


artifacts = {
    'fold_scores': OUTPUT_DIR / 'experiment_A_fold_scores.csv',
    'fold_predictions': OUTPUT_DIR / 'experiment_A_fold_predictions.csv',
    'split_manifest': OUTPUT_DIR / 'experiment_A_split_manifest.csv',
    'score_summary': OUTPUT_DIR / 'experiment_A_score_summary.csv',
    'paired_fold_differences': OUTPUT_DIR / 'experiment_A_paired_fold_differences.csv',
    'paired_summary': OUTPUT_DIR / 'experiment_A_paired_summary.csv',
    'material_error_analysis': OUTPUT_DIR / 'experiment_A_material_error_analysis.csv',
    'target_registry': OUTPUT_DIR / 'experiment_A_target_registry.csv',
    'family_manifest': OUTPUT_DIR / 'experiment_A_feature_family_manifest.csv',
    'feature_set_manifest': OUTPUT_DIR / 'experiment_A_feature_set_manifest.csv',
    'run_metadata': OUTPUT_DIR / 'experiment_A_run_metadata.json',
}

for frame, key in [
    (fold_scores, 'fold_scores'),
    (fold_predictions, 'fold_predictions'),
    (split_manifest, 'split_manifest'),
    (score_summary, 'score_summary'),
    (paired_fold_differences, 'paired_fold_differences'),
    (paired_summary, 'paired_summary'),
    (material_errors, 'material_error_analysis'),
    (target_registry, 'target_registry'),
    (family_manifest, 'family_manifest'),
    (set_manifest, 'feature_set_manifest'),
]:
    frame.to_csv(artifacts[key], index=False)

feature_manifest_payload = {
    'families': families,
    'feature_sets': FEATURE_SETS,
    'excluded_columns': sorted(EXCLUDED_COLUMNS),
}
with (OUTPUT_DIR / 'experiment_A_feature_manifest.json').open('w', encoding='utf-8') as handle:
    json.dump(feature_manifest_payload, handle, indent=2)

run_metadata = {
    'data_path': str(DATA_PATH),
    'data_sha256': sha256_file(DATA_PATH),
    'data_shape': list(df_raw.shape),
    'is_demo': IS_DEMO,
    'id_column': ID_COLUMN,
    'formula_column': FORMULA_COLUMN,
    'group_column': GROUP_COLUMN,
    'composition_backend': COMPOSITION_BACKEND,
    'config': asdict(CONFIG),
    'package_versions': {
        'python': sys.version.split()[0],
        'numpy': np.__version__,
        'pandas': pd.__version__,
        'scipy': package_version('scipy'),
        'scikit-learn': sklearn_version,
        'matplotlib': package_version('matplotlib'),
        'pymatgen': package_version('pymatgen'),
    },
}
with artifacts['run_metadata'].open('w', encoding='utf-8') as handle:
    json.dump(run_metadata, handle, indent=2)

artifact_table = pd.DataFrame([
    {'artifact': key, 'path': str(path.relative_to(REPO_ROOT)), 'exists': path.exists(), 'bytes': path.stat().st_size if path.exists() else 0}
    for key, path in artifacts.items()
] + [{
    'artifact': 'feature_manifest',
    'path': str((OUTPUT_DIR / 'experiment_A_feature_manifest.json').relative_to(REPO_ROOT)),
    'exists': (OUTPUT_DIR / 'experiment_A_feature_manifest.json').exists(),
    'bytes': (OUTPUT_DIR / 'experiment_A_feature_manifest.json').stat().st_size,
}])
display(artifact_table)



## Interpretation checklist before publishing results

1. **Replace demo data:** use a cohort large enough to support the number of groups, target classes, and requested folds.
2. **Add formation energy:** the current repository demo schema does not include formation energy per atom; request and persist that Materials Project field in the collection pipeline.
3. **Confirm label provenance:** prefer direct metallicity and synthesized-status fields over proxies when available.
4. **Confirm grouping:** reduced-formula grouping controls polymorph leakage; prototype or chemical-system grouping answers a harder extrapolation question.
5. **Inspect missingness:** if a layer family is mostly absent, distinguish a physically meaningful detector-negative value from a failed computation.
6. **Read paired effects, not rankings alone:** prioritize confidence intervals, fold win rates, and consistency across targets.
7. **Run a frozen nonlinear sensitivity model later:** reuse the exported split manifest exactly, so any change reflects model class rather than a new validation draw.

### What would support the Experiment A claim?

The strongest portfolio-ready result would be a positive Combined-vs-Baseline interval for multiple targets, with geometry and chemistry ablations clarifying *where* the signal originates, no dependence on a small number of groups, and materially improved out-of-fold errors for chemically diverse materials.
